# Fashion-MNIST | Optuna-Driven CNN Hyperparameter Optimization

**A tuning-first, reproducible deep-learning experiment built with PyTorch, Optuna, and Plotly.**

This notebook is designed around one central question: **which CNN architecture and training configuration gives the strongest validation performance on Fashion-MNIST?** Optuna is not a small add-on here; it drives the architecture, optimizer, learning rate, regularization, batch size, and label-smoothing decisions.

### Experiment design

1. Validate and explore the supplied `fashion-mnist_train.csv` and `fashion-mnist_test.csv` files.
2. Split only the training CSV into stratified training and validation partitions. The official test CSV stays untouched until final evaluation.
3. Run a reproducible Optuna study with a TPE sampler, Hyperband pruning, resumable SQLite storage, and a broader search space.
4. Inspect the search with interactive Plotly optimization-history, importance, parallel-coordinate, slice, contour, and intermediate-value charts.
5. Retrain the best configuration on the full training partition, then evaluate once on the official test data.

### Visualization and file policy

- **All graphs use Plotly** and remain interactive in the notebook.
- Graphs are saved into `images/` as interactive HTML and, when the static renderer is available, high-resolution PNG files.
- Model checkpoints, study database, trial tables, metrics, and prediction exports are saved into `artifacts/`.
- Notebook outputs intentionally show results, not generated file locations.
- The notebook contains exactly 100 Python code cells.

In [1]:
# Install missing visualization and experiment packages quietly.
import importlib.util
import subprocess
import sys

required_packages = {
    "optuna": "optuna",
    "plotly": "plotly",
    "sklearn": "scikit-learn",
    "kaleido": "kaleido==0.2.1",
}
missing_packages = [package for module, package in required_packages.items()
                    if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

In [2]:
# Core Python, data science, interactive visualization, and deep-learning imports.
import copy
import gc
import json
import math
import random
import warnings
from pathlib import Path

import joblib
import numpy as np
import optuna
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.metrics import (
    accuracy_score, auc, classification_report, confusion_matrix,
    f1_score, precision_recall_curve, precision_score, recall_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from torch.utils.data import DataLoader, Dataset, Subset

print("Core imports completed successfully.")

Core imports completed successfully.


In [3]:
# Keep notebook output readable and make Plotly rendering consistent.
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)
pd.set_option("display.precision", 4)

NOTEBOOK_TITLE = "Fashion-MNIST | Optuna Tuning Lab"
PLOTLY_TEMPLATE = "plotly_white"
PLOTLY_COLORS = ["#5B5FDE", "#00A6A6", "#F28E2B", "#E15759", "#59A14F",
                 "#B07AA1", "#76B7B2", "#EDC948", "#4E79A7", "#FF9DA7"]
print(NOTEBOOK_TITLE)

Fashion-MNIST | Optuna Tuning Lab


In [4]:
# Reproducibility settings for Python, NumPy, and PyTorch.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"Random seed: {SEED}")

Random seed: 42


In [5]:
# Create a clean output structure and centralize all Plotly figure exports.
PROJECT_ROOT = Path.cwd()
IMAGES_DIR = PROJECT_ROOT / "images"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


def style_plotly_figure(fig, title=None, height=620):
    """Apply one consistent, presentation-ready Plotly theme."""
    fig.update_layout(
        template=PLOTLY_TEMPLATE,
        title=dict(text=title or fig.layout.title.text, x=0.03, xanchor="left",
                   font=dict(size=21, color="#172554")),
        font=dict(family="Arial, sans-serif", size=12, color="#25324B"),
        paper_bgcolor="#FFFFFF",
        plot_bgcolor="#F7F9FC",
        colorway=PLOTLY_COLORS,
        height=height,
        margin=dict(l=60, r=35, t=85, b=65),
        hoverlabel=dict(bgcolor="#172554", font_color="#FFFFFF"),
        legend=dict(bgcolor="rgba(255,255,255,0.82)", bordercolor="#E5EAF3", borderwidth=1),
    )
    return fig


def save_plotly_figure(fig, filename, title=None, height=620, width=1400):
    """Save an interactive HTML chart and attempt a high-resolution PNG export."""
    filename = Path(filename).stem
    style_plotly_figure(fig, title=title, height=height)
    fig.write_html(
        IMAGES_DIR / f"{filename}.html",
        include_plotlyjs="cdn",
        full_html=True,
        auto_open=False,
    )
    try:
        fig.write_image(
            str(IMAGES_DIR / f"{filename}.png"),
            width=width,
            height=height,
            scale=2,
        )
    except Exception:
        # HTML remains available even when the local static-image engine is unavailable.
        pass
    display(fig)

print("Output folders and Plotly export helpers are ready.")

Output folders and Plotly export helpers are ready.


In [6]:
# Locate input CSVs in common notebook and Colab working locations.
def locate_dataset_file(filename: str) -> Path:
    search_roots = [
        Path.cwd(), Path.cwd() / "data", Path("/content"),
        Path("/content/data"), Path("/mnt/data"),
    ]
    for root in search_roots:
        candidate = root / filename
        if candidate.is_file():
            return candidate
    for root in (Path.cwd(), Path("/content")):
        if root.exists():
            matches = list(root.glob(f"**/{filename}"))
            if matches:
                return matches[0]
    raise FileNotFoundError(
        f"Could not locate {filename}. Keep the CSV beside this notebook or in a data folder."
    )

In [7]:
# Resolve the two supplied Fashion-MNIST CSV datasets.
TRAIN_CSV = locate_dataset_file("fashion-mnist_train.csv")
TEST_CSV = locate_dataset_file("fashion-mnist_test.csv")

# Do not display the resolved locations in notebook output.
assert TRAIN_CSV.is_file() and TEST_CSV.is_file()
print("Both Fashion-MNIST CSV files were found.")

Both Fashion-MNIST CSV files were found.


In [8]:
# Fashion-MNIST contains 28×28 grayscale images and ten clothing categories.
IMAGE_HEIGHT = 28
IMAGE_WIDTH = 28
NUM_PIXELS = IMAGE_HEIGHT * IMAGE_WIDTH
NUM_CLASSES = 10
PIXEL_COLUMNS = [f"pixel{i}" for i in range(1, NUM_PIXELS + 1)]
EXPECTED_COLUMNS = ["label", *PIXEL_COLUMNS]

CLASS_NAMES = {
    0: "T-shirt/top", 1: "Trouser", 2: "Pullover", 3: "Dress", 4: "Coat",
    5: "Sandal", 6: "Shirt", 7: "Sneaker", 8: "Bag", 9: "Ankle boot",
}
CLASS_IDS = list(CLASS_NAMES.keys())
print(f"Expected image dimensions: {IMAGE_HEIGHT} × {IMAGE_WIDTH}")

Expected image dimensions: 28 × 28


In [9]:
# Load the training split from the supplied CSV.
train_df = pd.read_csv(TRAIN_CSV)
print(f"Training rows loaded: {len(train_df):,}")

Training rows loaded: 60,000


In [10]:
# Load the official test split separately; it will only be used for final evaluation.
test_df = pd.read_csv(TEST_CSV)
print(f"Official test rows loaded: {len(test_df):,}")

Official test rows loaded: 10,000


## 2. Data validation and exploratory analysis

The source schema should contain one `label` column and 784 pixel columns. Exploratory plots are saved automatically while remaining visible inline.

In [11]:
# Inspect the dimensions of both source tables.
shape_summary = pd.DataFrame(
    {"Dataset": ["Training CSV", "Official test CSV"],
     "Rows": [len(train_df), len(test_df)],
     "Columns": [train_df.shape[1], test_df.shape[1]]}
)
display(shape_summary)

,Dataset,Rows,Columns
0,Training CSV,60000,785
1,Official test CSV,10000,785


In [12]:
# Validate the schema before converting any data into tensors.
for dataset_name, frame in (("training", train_df), ("test", test_df)):
    missing = sorted(set(EXPECTED_COLUMNS) - set(frame.columns))
    if missing:
        raise ValueError(f"The {dataset_name} CSV is missing required columns.")

print("Schema validation passed: label plus 784 pixel columns.")

Schema validation passed: label plus 784 pixel columns.


In [13]:
# Validate label values and normalize their data type.
for frame in (train_df, test_df):
    frame["label"] = frame["label"].astype("int64")
    invalid_labels = sorted(set(frame["label"].unique()) - set(CLASS_IDS))
    if invalid_labels:
        raise ValueError("Unexpected class labels were found in the dataset.")

print("Label validation passed for all ten classes.")

Label validation passed for all ten classes.


In [14]:
# Check missing values and nulls in the source tables.
integrity_summary = pd.DataFrame({
    "Dataset": ["Training", "Official test"],
    "Missing cells": [int(train_df.isna().sum().sum()), int(test_df.isna().sum().sum())],
    "Duplicate rows": [int(train_df.duplicated().sum()), int(test_df.duplicated().sum())],
})
display(integrity_summary)

,Dataset,Missing cells,Duplicate rows
0,Training,0,43
1,Official test,0,1


In [15]:
# Confirm pixel values follow the expected 8-bit grayscale range.
train_pixel_min = int(train_df[PIXEL_COLUMNS].min().min())
train_pixel_max = int(train_df[PIXEL_COLUMNS].max().max())
test_pixel_min = int(test_df[PIXEL_COLUMNS].min().min())
test_pixel_max = int(test_df[PIXEL_COLUMNS].max().max())

pixel_range_summary = pd.DataFrame({
    "Dataset": ["Training", "Official test"],
    "Minimum pixel": [train_pixel_min, test_pixel_min],
    "Maximum pixel": [train_pixel_max, test_pixel_max],
})
display(pixel_range_summary)
assert 0 <= min(train_pixel_min, test_pixel_min)
assert max(train_pixel_max, test_pixel_max) <= 255

,Dataset,Minimum pixel,Maximum pixel
0,Training,0,255
1,Official test,0,255


In [16]:
# Summarize the numeric pixel values without printing any input file locations.
pixel_summary_sample = train_df[PIXEL_COLUMNS].to_numpy(dtype=np.uint8)[::100].ravel()
train_pixel_summary = pd.Series(pixel_summary_sample).describe(percentiles=[0.25, 0.5, 0.75])
display(train_pixel_summary.to_frame(name="Training pixel intensity"))

,Training pixel intensity
count,470400.0000
mean,74.2172
std,90.8960
min,0.0000
25%,0.0000
50%,1.0000
75%,167.0000
max,255.0000


In [17]:
# Build class-frequency tables for both supplied splits.
train_class_counts = train_df["label"].value_counts().reindex(CLASS_IDS, fill_value=0)
test_class_counts = test_df["label"].value_counts().reindex(CLASS_IDS, fill_value=0)
class_distribution = pd.DataFrame({
    "Class ID": CLASS_IDS,
    "Class": [CLASS_NAMES[i] for i in CLASS_IDS],
    "Training": train_class_counts.values,
    "Official test": test_class_counts.values,
})
display(class_distribution)

,Class ID,Class,Training,Official test
0,0,T-shirt/top,6000,1000
1,1,Trouser,6000,1000
2,2,Pullover,6000,1000
3,3,Dress,6000,1000
4,4,Coat,6000,1000
5,5,Sandal,6000,1000
6,6,Shirt,6000,1000
7,7,Sneaker,6000,1000
8,8,Bag,6000,1000
9,9,Ankle boot,6000,1000


In [18]:
# Plot training examples per class with a consistent, distinct color for each category.
class_plot_df = class_distribution[["Class", "Training"]].copy()
fig = px.bar(
    class_plot_df, x="Class", y="Training", color="Class",
    color_discrete_sequence=PLOTLY_COLORS, text_auto=".3s",
    labels={"Training": "Number of images", "Class": "Fashion category"},
)
fig.update_traces(marker_line_width=0, textposition="outside")
fig.update_xaxes(tickangle=-25, showgrid=False)
fig.update_yaxes(gridcolor="#E8EDF5")
save_plotly_figure(fig, "01_training_class_distribution", title="Training Set | Images per Class", height=560)

In [19]:
# Compare the class balance in the training and official test datasets.
plot_distribution = class_distribution.melt(
    id_vars=["Class ID", "Class"], var_name="Split", value_name="Images"
)
fig = px.bar(
    plot_distribution, x="Class", y="Images", color="Split", barmode="group",
    color_discrete_map={"Training": "#5B5FDE", "Official test": "#00A6A6"},
    labels={"Images": "Number of images", "Class": "Fashion category"},
)
fig.update_layout(legend_title_text="Dataset split")
fig.update_xaxes(tickangle=-25, showgrid=False)
fig.update_yaxes(gridcolor="#E8EDF5")
save_plotly_figure(fig, "02_train_test_class_comparison", title="Class Distribution | Training vs Official Test", height=560)

In [20]:
# Explore the distribution of pixel intensities across the training images.
pixel_sample = train_df[PIXEL_COLUMNS].to_numpy(dtype=np.uint8)[::20].ravel()
fig = px.histogram(
    x=pixel_sample, nbins=48, marginal="box", opacity=0.88,
    color_discrete_sequence=["#7C4DFF"],
    labels={"x": "Grayscale intensity (0–255)", "y": "Pixel count"},
)
fig.update_layout(bargap=0.04)
save_plotly_figure(fig, "03_pixel_intensity_distribution", title="Pixel Intensity Distribution", height=560)

In [21]:
# Display a random montage of training images with human-readable labels.
rng = np.random.default_rng(SEED)
random_indices = rng.choice(len(train_df), size=25, replace=False)
fig = make_subplots(rows=5, cols=5, subplot_titles=[
    CLASS_NAMES[int(train_df.iloc[int(idx)]["label"])] for idx in random_indices
], vertical_spacing=0.055, horizontal_spacing=0.035)
for position, idx in enumerate(random_indices):
    row, col = divmod(position, 5)
    image = train_df.iloc[int(idx)][PIXEL_COLUMNS].to_numpy(dtype=np.uint8).reshape(28, 28)
    fig.add_trace(go.Heatmap(z=image, colorscale="Magma", zmin=0, zmax=255,
                             showscale=False, hoverinfo="skip"), row=row + 1, col=col + 1)
    fig.update_xaxes(visible=False, row=row + 1, col=col + 1)
    fig.update_yaxes(visible=False, autorange="reversed", row=row + 1, col=col + 1)
save_plotly_figure(fig, "04_random_training_samples", title="Random Fashion-MNIST Training Samples", height=900, width=1100)

In [22]:
# Show one representative image from each class.
fig = make_subplots(rows=2, cols=5, subplot_titles=[CLASS_NAMES[i] for i in CLASS_IDS],
                    vertical_spacing=0.10, horizontal_spacing=0.05)
for position, class_id in enumerate(CLASS_IDS):
    row, col = divmod(position, 5)
    sample_row = train_df.loc[train_df["label"] == class_id].iloc[0]
    image = sample_row[PIXEL_COLUMNS].to_numpy(dtype=np.uint8).reshape(28, 28)
    fig.add_trace(go.Heatmap(z=image, colorscale="Viridis", zmin=0, zmax=255,
                             showscale=False, hoverinfo="skip"), row=row + 1, col=col + 1)
    fig.update_xaxes(visible=False, row=row + 1, col=col + 1)
    fig.update_yaxes(visible=False, autorange="reversed", row=row + 1, col=col + 1)
save_plotly_figure(fig, "05_class_reference_images", title="One Reference Image per Fashion Category", height=650, width=1400)

In [23]:
# Calculate and visualize the average training image.
mean_image = train_df[PIXEL_COLUMNS].to_numpy(dtype=np.float32).mean(axis=0).reshape(28, 28)
fig = go.Figure(go.Heatmap(z=mean_image, colorscale="Magma", colorbar=dict(title="Mean intensity")))
fig.update_yaxes(autorange="reversed", scaleanchor="x")
save_plotly_figure(fig, "06_mean_training_image", title="Mean Pixel Intensity Across Training Images", height=560, width=800)

In [24]:
# Visualize pixel-level variability using the standard deviation image.
std_image = train_df[PIXEL_COLUMNS].to_numpy(dtype=np.float32).std(axis=0).reshape(28, 28)
fig = go.Figure(go.Heatmap(z=std_image, colorscale="Plasma", colorbar=dict(title="Std. deviation")))
fig.update_yaxes(autorange="reversed", scaleanchor="x")
save_plotly_figure(fig, "07_pixel_variability_map", title="Pixel-Level Standard Deviation", height=560, width=800)

In [25]:
# Compare average silhouettes for all ten categories.
class_mean_images = {}
for class_id in CLASS_IDS:
    values = train_df.loc[train_df["label"] == class_id, PIXEL_COLUMNS].to_numpy(dtype=np.float32)
    class_mean_images[class_id] = values.mean(axis=0).reshape(28, 28)
fig = make_subplots(rows=2, cols=5, subplot_titles=[CLASS_NAMES[i] for i in CLASS_IDS],
                    vertical_spacing=0.10, horizontal_spacing=0.05)
for position, class_id in enumerate(CLASS_IDS):
    row, col = divmod(position, 5)
    fig.add_trace(go.Heatmap(z=class_mean_images[class_id], colorscale="Inferno",
                             showscale=False, hoverinfo="skip"), row=row + 1, col=col + 1)
    fig.update_xaxes(visible=False, row=row + 1, col=col + 1)
    fig.update_yaxes(visible=False, autorange="reversed", row=row + 1, col=col + 1)
save_plotly_figure(fig, "08_class_average_images", title="Average Image by Class", height=680, width=1400)

In [26]:
# Examine the intensity distribution for a single randomly selected image.
sample_image = train_df.iloc[int(random_indices[0])][PIXEL_COLUMNS].to_numpy(dtype=np.uint8)
fig = px.histogram(
    x=sample_image, nbins=32, color_discrete_sequence=["#00A6A6"],
    labels={"x": "Pixel intensity", "y": "Pixel count"},
)
fig.update_layout(bargap=0.05)
save_plotly_figure(fig, "09_single_image_intensity_histogram", title="Pixel Intensities in One Sample Image", height=520)

In [27]:
# Check duplicate rows as a data-quality signal.
duplicate_summary = pd.DataFrame({
    "Dataset": ["Training", "Official test"],
    "Duplicate rows": [int(train_df.duplicated().sum()), int(test_df.duplicated().sum())],
    "Duplicate share (%)": [100 * train_df.duplicated().mean(), 100 * test_df.duplicated().mean()],
})
display(duplicate_summary.round(3))

,Dataset,Duplicate rows,Duplicate share (%)
0,Training,43,0.072
1,Official test,1,0.010


## 3. Train/validation preparation

A stratified validation subset is carved from the training CSV. Pixel normalization statistics are estimated from training images only and reused for validation and test images.

In [28]:
# Create a stratified validation split from the training CSV only.
all_train_indices = np.arange(len(train_df))
train_indices, validation_indices = train_test_split(
    all_train_indices,
    test_size=0.15,
    random_state=SEED,
    stratify=train_df["label"].to_numpy(),
)
print(f"Training examples: {len(train_indices):,} | Validation examples: {len(validation_indices):,}")

Training examples: 51,000 | Validation examples: 9,000


In [29]:
# Convert the CSV tables into compact uint8 arrays and integer labels.
X_train_raw = train_df.iloc[train_indices][PIXEL_COLUMNS].to_numpy(dtype=np.uint8)
y_train = train_df.iloc[train_indices]["label"].to_numpy(dtype=np.int64)
X_val_raw = train_df.iloc[validation_indices][PIXEL_COLUMNS].to_numpy(dtype=np.uint8)
y_val = train_df.iloc[validation_indices]["label"].to_numpy(dtype=np.int64)
X_test_raw = test_df[PIXEL_COLUMNS].to_numpy(dtype=np.uint8)
y_test = test_df["label"].to_numpy(dtype=np.int64)

print(f"Array shapes — train: {X_train_raw.shape}, validation: {X_val_raw.shape}, test: {X_test_raw.shape}")

Array shapes — train: (51000, 784), validation: (9000, 784), test: (10000, 784)


In [30]:
# Estimate normalization statistics from training images only.
PIXEL_MEAN = float(X_train_raw.mean(dtype=np.float64) / 255.0)
PIXEL_STD = float(X_train_raw.std(dtype=np.float64) / 255.0)
PIXEL_STD = max(PIXEL_STD, 1e-6)
print(f"Training pixel mean: {PIXEL_MEAN:.4f} | standard deviation: {PIXEL_STD:.4f}")

# Release the original pandas tables to reduce memory pressure.
del train_df, test_df
gc.collect()

Training pixel mean: 0.2860 | standard deviation: 0.3527


6711

In [31]:
# Dataset class: converts uint8 rows to normalized tensors and applies light translation augmentation.
class FashionCSVDataset(Dataset):
    def __init__(self, images, labels, mean, std, augment=False):
        self.images = images
        self.labels = labels
        self.mean = float(mean)
        self.std = float(std)
        self.augment = bool(augment)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image = torch.as_tensor(self.images[index], dtype=torch.float32).view(1, 28, 28) / 255.0
        if self.augment:
            padded = F.pad(image, (2, 2, 2, 2), mode="reflect")
            top = int(torch.randint(0, 5, (1,)).item())
            left = int(torch.randint(0, 5, (1,)).item())
            image = padded[:, top:top + 28, left:left + 28]
        image = (image - self.mean) / self.std
        label = torch.tensor(int(self.labels[index]), dtype=torch.long)
        return image, label

In [32]:
# Instantiate training, validation, and official test datasets.
train_dataset = FashionCSVDataset(X_train_raw, y_train, PIXEL_MEAN, PIXEL_STD, augment=True)
train_eval_dataset = FashionCSVDataset(X_train_raw, y_train, PIXEL_MEAN, PIXEL_STD, augment=False)
val_dataset = FashionCSVDataset(X_val_raw, y_val, PIXEL_MEAN, PIXEL_STD, augment=False)
test_dataset = FashionCSVDataset(X_test_raw, y_test, PIXEL_MEAN, PIXEL_STD, augment=False)
print("Tensor datasets are ready.")

Tensor datasets are ready.


In [33]:
# Configure DataLoaders with conservative worker settings for notebook portability.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128
NUM_WORKERS = 0
PIN_MEMORY = DEVICE.type == "cuda"

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
train_eval_loader = DataLoader(train_eval_dataset, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
print(f"Compute device: {DEVICE}")

Compute device: cpu


In [34]:
# Inspect the shape, type, and label range of one batch.
batch_images, batch_labels = next(iter(train_loader))
batch_summary = pd.DataFrame({
    "Property": ["Image batch shape", "Image tensor dtype", "Label batch shape",
                 "Minimum normalized pixel", "Maximum normalized pixel"],
    "Value": [str(tuple(batch_images.shape)), str(batch_images.dtype),
              str(tuple(batch_labels.shape)), float(batch_images.min()), float(batch_images.max())],
})
display(batch_summary)

,Property,Value
0,Image batch shape,"(128, 1, 28, 28)"
1,Image tensor dtype,torch.float32
2,Label batch shape,"(128,)"
3,Minimum normalized pixel,-0.8109
4,Maximum normalized pixel,2.0247


In [35]:
# Visualize a batch after normalization and random translation augmentation.
fig = make_subplots(rows=4, cols=6, subplot_titles=[CLASS_NAMES[int(v)] for v in batch_labels[:24]],
                    vertical_spacing=0.06, horizontal_spacing=0.035)
for idx in range(24):
    row, col = divmod(idx, 6)
    image = batch_images[idx, 0].cpu().numpy() * PIXEL_STD + PIXEL_MEAN
    fig.add_trace(go.Heatmap(z=np.clip(image, 0, 1), colorscale="Gray", zmin=0, zmax=1,
                             showscale=False, hoverinfo="skip"), row=row + 1, col=col + 1)
    fig.update_xaxes(visible=False, row=row + 1, col=col + 1)
    fig.update_yaxes(visible=False, autorange="reversed", row=row + 1, col=col + 1)
save_plotly_figure(fig, "10_augmented_training_batch", title="Augmented Training Batch", height=760, width=1300)

In [36]:
# Calculate optional class weights; the training data is expected to be close to balanced.
class_counts_train = np.bincount(y_train, minlength=NUM_CLASSES)
class_weights_np = len(y_train) / (NUM_CLASSES * np.maximum(class_counts_train, 1))
class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=DEVICE)
class_weight_table = pd.DataFrame({
    "Class": [CLASS_NAMES[i] for i in CLASS_IDS],
    "Training count": class_counts_train,
    "Loss weight": class_weights_np,
})
display(class_weight_table.round(4))

,Class,Training count,Loss weight
0,T-shirt/top,5100,1.0
1,Trouser,5100,1.0
2,Pullover,5100,1.0
3,Dress,5100,1.0
4,Coat,5100,1.0
5,Sandal,5100,1.0
6,Shirt,5100,1.0
7,Sneaker,5100,1.0
8,Bag,5100,1.0
9,Ankle boot,5100,1.0


## 4. CNN architecture and training utilities

The network stacks convolution, batch normalization, activation, pooling, and dropout blocks. Adaptive pooling keeps the classifier compatible with different CNN depths.

In [37]:
# Baseline reference configuration and final-training budget.
BASE_CONFIG = {
    "num_blocks": 2,
    "base_channels": 32,
    "kernel_size": 3,
    "dropout": 0.30,
}
BASELINE_EPOCHS = 4
FINAL_EPOCHS = 24
EARLY_STOPPING_PATIENCE = 6
print("Reference architecture:", BASE_CONFIG)

Reference architecture: {'num_blocks': 2, 'base_channels': 32, 'kernel_size': 3, 'dropout': 0.3}


In [38]:
# Dynamic CNN with stacked convolution blocks and adaptive global pooling.
class FashionCNN(nn.Module):
    def __init__(self, num_blocks=2, base_channels=32, kernel_size=3, dropout=0.3):
        super().__init__()
        if num_blocks not in (1, 2, 3):
            raise ValueError("num_blocks must be 1, 2, or 3")
        padding = kernel_size // 2
        layers = []
        in_channels = 1
        self.target_layer = None
        for block_index in range(num_blocks):
            out_channels = base_channels * (2 ** block_index)
            conv_one = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding, bias=False)
            conv_two = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
            layers.extend([
                conv_one, nn.BatchNorm2d(out_channels), nn.ReLU(inplace=False),
                conv_two, nn.BatchNorm2d(out_channels), nn.ReLU(inplace=False),
                nn.MaxPool2d(2), nn.Dropout2d(dropout * 0.5),
            ])
            self.target_layer = conv_two
            in_channels = out_channels
        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_channels, 128),
            nn.ReLU(inplace=False),
            nn.Dropout(dropout),
            nn.Linear(128, NUM_CLASSES),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

In [39]:
# Create the baseline network and move it to the selected compute device.
baseline_model = FashionCNN(**BASE_CONFIG).to(DEVICE)
print(f"Baseline model created on {DEVICE.type.upper()}.")

Baseline model created on CPU.


In [40]:
# Count trainable parameters and review the model's high-level components.
trainable_parameters = sum(parameter.numel() for parameter in baseline_model.parameters()
                           if parameter.requires_grad)
model_overview = pd.DataFrame({
    "Component": ["Convolutional feature extractor", "Adaptive average pooling", "Classifier"],
    "Role": ["Learn spatial patterns", "Compress feature maps", "Map features to 10 classes"],
})
display(model_overview)
print(f"Trainable parameters: {trainable_parameters:,}")

,Component,Role
0,Convolutional feature extractor,Learn spatial patterns
1,Adaptive average pooling,Compress feature maps
2,Classifier,Map features to 10 classes


Trainable parameters: 74,794


In [41]:
# Forward-pass smoke test: output must contain ten class logits per image.
with torch.no_grad():
    sample_logits = baseline_model(batch_images[:4].to(DEVICE))
assert tuple(sample_logits.shape) == (4, NUM_CLASSES)
print("Forward-pass validation passed.")

Forward-pass validation passed.


In [42]:
# Shared loss and optimizer factories keep baseline, tuning, and final training consistent.
criterion = nn.CrossEntropyLoss(weight=class_weights)


def create_optimizer(model, learning_rate=1e-3, weight_decay=1e-4, optimizer_name="AdamW"):
    optimizer_types = {
        "AdamW": optim.AdamW,
        "Adam": optim.Adam,
        "RMSprop": optim.RMSprop,
        "SGD": optim.SGD,
    }
    if optimizer_name not in optimizer_types:
        raise ValueError("Unsupported optimizer configuration.")
    optimizer_kwargs = {"lr": float(learning_rate), "weight_decay": float(weight_decay)}
    if optimizer_name == "SGD":
        optimizer_kwargs.update({"momentum": 0.9, "nesterov": True})
    return optimizer_types[optimizer_name](model.parameters(), **optimizer_kwargs)

In [43]:
# Prediction and metric helpers used throughout the evaluation workflow.
def predict_loader(model, loader):
    model.eval()
    all_labels, all_predictions, all_probabilities = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(DEVICE))
            probabilities = torch.softmax(logits, dim=1)
            all_labels.append(labels.cpu().numpy())
            all_predictions.append(probabilities.argmax(dim=1).cpu().numpy())
            all_probabilities.append(probabilities.cpu().numpy())
    return (np.concatenate(all_labels), np.concatenate(all_predictions),
            np.concatenate(all_probabilities))

def classification_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }

In [44]:
# One complete training epoch with weighted loss and online accuracy.
def train_one_epoch(model, loader, optimizer, loss_function):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = loss_function(logits, labels)
        loss.backward()
        optimizer.step()
        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += batch_size
    return running_loss / max(total, 1), correct / max(total, 1)

In [45]:
# Evaluate a model without updating weights.
def evaluate_one_epoch(model, loader, loss_function):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            logits = model(images)
            loss = loss_function(logits, labels)
            batch_size = labels.size(0)
            running_loss += loss.item() * batch_size
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += batch_size
    return {"loss": running_loss / max(total, 1), "accuracy": correct / max(total, 1)}

In [46]:
# Training loop with configurable loss, learning-rate scheduling, and early stopping.
def fit_model(model, training_loader, validation_loader, optimizer, epochs, patience=5, loss_function=None):
    active_loss = loss_function if loss_function is not None else criterion
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )
    history = []
    best_score = -1.0
    best_state = copy.deepcopy(model.state_dict())
    epochs_without_improvement = 0
    for epoch in range(1, int(epochs) + 1):
        train_loss, train_accuracy = train_one_epoch(model, training_loader, optimizer, active_loss)
        validation_result = evaluate_one_epoch(model, validation_loader, active_loss)
        scheduler.step(validation_result["accuracy"])
        history.append({
            "epoch": epoch, "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": validation_result["loss"],
            "val_accuracy": validation_result["accuracy"],
            "learning_rate": optimizer.param_groups[0]["lr"],
        })
        if validation_result["accuracy"] > best_score:
            best_score = validation_result["accuracy"]
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        print(f"Epoch {epoch:02d}/{epochs} | train acc {train_accuracy:.4f} "
              f"| val acc {validation_result['accuracy']:.4f} "
              f"| val loss {validation_result['loss']:.4f}")
        if epochs_without_improvement >= patience:
            print("Early stopping triggered.")
            break
    model.load_state_dict(best_state)
    return pd.DataFrame(history), best_score

In [47]:
# Train the baseline model to establish a reference point before hyperparameter search.
baseline_optimizer = create_optimizer(baseline_model, learning_rate=1e-3,
                                       weight_decay=1e-4, optimizer_name="AdamW")
baseline_history, baseline_best_val_accuracy = fit_model(
    baseline_model, train_loader, val_loader, baseline_optimizer,
    epochs=BASELINE_EPOCHS, patience=BASELINE_EPOCHS + 1,
)
print(f"Best baseline validation accuracy: {baseline_best_val_accuracy:.4f}")

Epoch 01/4 | train acc 0.6487 | val acc 0.7638 | val loss 0.6084
Epoch 02/4 | train acc 0.7677 | val acc 0.8168 | val loss 0.4746
Epoch 03/4 | train acc 0.7979 | val acc 0.8347 | val loss 0.4527
Epoch 04/4 | train acc 0.8159 | val acc 0.8494 | val loss 0.4058
Best baseline validation accuracy: 0.8494


In [48]:
# Inspect baseline epoch-level metrics in a compact table.
display(baseline_history.round(4))

,epoch,train_loss,train_accuracy,val_loss,val_accuracy,learning_rate
0,1,0.9431,0.6487,0.6084,0.7638,0.001
1,2,0.6169,0.7677,0.4746,0.8168,0.001
2,3,0.5453,0.7979,0.4527,0.8347,0.001
3,4,0.4981,0.8159,0.4058,0.8494,0.001


In [49]:
# Plot baseline loss and accuracy curves with interactive hover details.
fig = make_subplots(rows=1, cols=2, subplot_titles=["Cross-Entropy Loss", "Accuracy"])
for name, column, color in [("Train", "train_loss", "#5B5FDE"), ("Validation", "val_loss", "#00A6A6")]:
    fig.add_trace(go.Scatter(x=baseline_history["epoch"], y=baseline_history[column],
                             mode="lines+markers", name=f"{name} loss", line=dict(color=color, width=3)), row=1, col=1)
for name, column, color in [("Train", "train_accuracy", "#5B5FDE"), ("Validation", "val_accuracy", "#00A6A6")]:
    fig.add_trace(go.Scatter(x=baseline_history["epoch"], y=baseline_history[column],
                             mode="lines+markers", name=f"{name} accuracy", line=dict(color=color, width=3)), row=1, col=2)
fig.update_xaxes(title_text="Epoch", row=1, col=1)
fig.update_xaxes(title_text="Epoch", row=1, col=2)
fig.update_yaxes(title_text="Loss", row=1, col=1)
fig.update_yaxes(title_text="Accuracy", range=[0, 1], row=1, col=2)
save_plotly_figure(fig, "11_baseline_training_curves", title="Baseline CNN Training Diagnostics", height=560, width=1350)

In [50]:
# Evaluate the baseline on validation data only.
baseline_y_true, baseline_y_pred, baseline_probabilities = predict_loader(baseline_model, val_loader)
baseline_metrics = classification_metrics(baseline_y_true, baseline_y_pred)
display(pd.DataFrame([baseline_metrics]).T.rename(columns={0: "Baseline validation"}))

,Baseline validation
accuracy,0.8494
macro_precision,0.8598
macro_recall,0.8494
macro_f1,0.8522
weighted_f1,0.8522


In [51]:
# Plot the baseline validation confusion matrix as an interactive heatmap.
baseline_cm = confusion_matrix(baseline_y_true, baseline_y_pred, labels=CLASS_IDS)
class_axis_labels = [CLASS_NAMES[i] for i in CLASS_IDS]
fig = go.Figure(go.Heatmap(
    z=baseline_cm, x=class_axis_labels, y=class_axis_labels,
    colorscale="Blues", text=baseline_cm, texttemplate="%{text}",
    colorbar=dict(title="Samples"), hovertemplate="True: %{y}<br>Predicted: %{x}<br>Count: %{z}<extra></extra>"
))
fig.update_layout(xaxis_title="Predicted class", yaxis_title="True class")
fig.update_xaxes(tickangle=-30)
fig.update_yaxes(autorange="reversed")
save_plotly_figure(fig, "12_baseline_validation_confusion_matrix", title="Baseline CNN | Validation Confusion Matrix", height=760, width=1100)

In [52]:
# Preserve the baseline checkpoint as a reference artifact.
torch.save({"state_dict": baseline_model.state_dict(), "config": BASE_CONFIG,
            "validation_metrics": baseline_metrics},
           ARTIFACTS_DIR / "baseline_checkpoint.pth")
print("Baseline checkpoint saved.")

Baseline checkpoint saved.


## 5. Optuna hyperparameter optimization — the core experiment

This is the main section of the notebook. Each trial trains a candidate CNN and is scored on the **stratified validation partition** using **macro F1**. Accuracy and validation loss are also logged as trial attributes for deeper analysis.

### Search dimensions

Optuna explores CNN depth, channel width, first-kernel size, dropout, optimizer family, learning rate, weight decay, batch size, and label smoothing. A seeded **TPE sampler** proposes configurations; **Hyperband pruning** stops weak trials early based on intermediate validation results.

### Experiment controls

The default budget is hardware-aware: a larger search on CUDA and a smaller starter budget on CPU. Increase `OPTUNA_TRIALS` for a more exhaustive search. The SQLite study database lets an interrupted run resume without losing completed trials. The held-out official test set is never used to select hyperparameters.

In [53]:
# Configure a meaningful, hardware-aware Optuna search budget and make the search space explicit.
OPTUNA_TRIALS = 30 if DEVICE.type == "cuda" else 12
TUNING_EPOCHS = 8 if DEVICE.type == "cuda" else 4
TUNING_FRACTION = 0.35
TUNING_VALIDATION_FRACTION = 0.50
OPTUNA_BATCH_CHOICES = [64, 128, 256]
OPTUNA_STUDY_NAME = f"fashion_mnist_cnn_tpe_seed_{SEED}"
OPTUNA_DB_PATH = ARTIFACTS_DIR / "optuna_study.db"

search_space = pd.DataFrame([
    ("CNN depth", "num_blocks", "1–3 convolution blocks"),
    ("Channel width", "base_channels", "16 / 24 / 32 / 48"),
    ("First kernel size", "kernel_size", "3 / 5"),
    ("Dropout", "dropout", "0.10–0.50"),
    ("Optimizer", "optimizer", "AdamW / Adam / RMSprop / SGD"),
    ("Learning rate", "learning_rate", "1e-4–5e-3 (log scale)"),
    ("Weight decay", "weight_decay", "1e-7–1e-2 (log scale)"),
    ("Batch size", "batch_size", "64 / 128 / 256"),
    ("Label smoothing", "label_smoothing", "0.00–0.15"),
], columns=["Hyperparameter", "Optuna parameter", "Search range"])
display(search_space)
print(f"Search target: {OPTUNA_TRIALS} trials | up to {TUNING_EPOCHS} epochs per trial")
print("Objective: maximize validation macro F1")

,Hyperparameter,Optuna parameter,Search range
0,CNN depth,num_blocks,1–3 convolution blocks
1,Channel width,base_channels,16 / 24 / 32 / 48
2,First kernel size,kernel_size,3 / 5
3,Dropout,dropout,0.10–0.50
4,Optimizer,optimizer,AdamW / Adam / RMSprop / SGD
5,Learning rate,learning_rate,1e-4–5e-3 (log scale)
6,Weight decay,weight_decay,1e-7–1e-2 (log scale)
7,Batch size,batch_size,64 / 128 / 256
8,Label smoothing,label_smoothing,0.00–0.15


Search target: 12 trials | up to 4 epochs per trial
Objective: maximize validation macro F1


In [54]:
# Build stratified tuning subsets; final training still uses the full training partition.
tune_train_indices, _ = train_test_split(
    np.arange(len(y_train)), train_size=TUNING_FRACTION,
    random_state=SEED, stratify=y_train,
)
tune_val_indices, _ = train_test_split(
    np.arange(len(y_val)), train_size=TUNING_VALIDATION_FRACTION,
    random_state=SEED, stratify=y_val,
)
tune_train_subset = Subset(train_dataset, tune_train_indices.tolist())
tune_val_subset = Subset(val_dataset, tune_val_indices.tolist())
print(f"Tuning subset sizes: {len(tune_train_subset):,} training / {len(tune_val_subset):,} validation")

Tuning subset sizes: 17,850 training / 4,500 validation


In [55]:
# Optuna objective: evaluate architecture and optimization hyperparameters using validation macro F1.
def objective(trial):
    trial_seed = SEED + int(trial.number)
    random.seed(trial_seed)
    np.random.seed(trial_seed)
    torch.manual_seed(trial_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(trial_seed)

    parameters = {
        "num_blocks": trial.suggest_int("num_blocks", 1, 3),
        "base_channels": trial.suggest_categorical("base_channels", [16, 24, 32, 48]),
        "kernel_size": trial.suggest_categorical("kernel_size", [3, 5]),
        "dropout": trial.suggest_float("dropout", 0.10, 0.50),
    }
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-7, 1e-2, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", ["AdamW", "Adam", "RMSprop", "SGD"])
    batch_size = trial.suggest_categorical("batch_size", OPTUNA_BATCH_CHOICES)
    label_smoothing = trial.suggest_float("label_smoothing", 0.0, 0.15)

    generator = torch.Generator().manual_seed(trial_seed)
    local_train_loader = DataLoader(
        tune_train_subset, batch_size=batch_size, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, generator=generator,
    )
    local_val_loader = DataLoader(
        tune_val_subset, batch_size=batch_size, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )

    model = None
    try:
        model = FashionCNN(**parameters).to(DEVICE)
        optimizer = create_optimizer(model, learning_rate, weight_decay, optimizer_name)
        trial_loss = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=2
        )

        best_macro_f1 = -1.0
        best_accuracy = 0.0
        best_loss = float("inf")
        best_epoch = 0
        stale_epochs = 0

        for epoch in range(1, TUNING_EPOCHS + 1):
            train_one_epoch(model, local_train_loader, optimizer, trial_loss)
            validation_result = evaluate_one_epoch(model, local_val_loader, trial_loss)
            y_val_trial, y_pred_trial, _ = predict_loader(model, local_val_loader)
            val_macro_f1 = float(f1_score(y_val_trial, y_pred_trial, average="macro", zero_division=0))
            scheduler.step(validation_result["loss"])

            trial.report(val_macro_f1, step=epoch)
            if val_macro_f1 > best_macro_f1:
                best_macro_f1 = val_macro_f1
                best_accuracy = float(validation_result["accuracy"])
                best_loss = float(validation_result["loss"])
                best_epoch = epoch
                stale_epochs = 0
            else:
                stale_epochs += 1

            trial.set_user_attr("best_validation_accuracy", best_accuracy)
            trial.set_user_attr("best_validation_loss", best_loss)
            trial.set_user_attr("best_epoch", best_epoch)
            trial.set_user_attr("parameter_count", int(sum(p.numel() for p in model.parameters())))

            if trial.should_prune():
                raise optuna.TrialPruned()
            if stale_epochs >= 3:
                break

        return best_macro_f1
    finally:
        if model is not None:
            del model
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

In [56]:
import os
import sys
import subprocess

# Force SQLAlchemy to install without compiled C extensions.
env = os.environ.copy()
env["DISABLE_SQLALCHEMY_CEXT"] = "1"

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--force-reinstall",
    "--no-cache-dir",
    "--no-binary=SQLAlchemy",
    "SQLAlchemy"
], env=env)

print("SQLAlchemy reinstalled without compiled extensions.")
print("Please restart the Jupyter kernel.")
import sys

# Create a persistent TPE study with Hyperband pruning and resumable SQLite storage.
optuna.logging.set_verbosity(optuna.logging.WARNING)
OPTUNA_STORAGE = f"sqlite:///{OPTUNA_DB_PATH.resolve().as_posix()}"
# Work around Windows App Control blocking the compiled immutabledict DLL.
try:
    import immutabledict  # noqa: F401
except Exception:
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--force-reinstall", "--no-cache-dir", "immutabledict"
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

# Reinstall without a native wheel and fall back to a pure-Python shim when
# Windows App Control blocks the compiled immutabledict DLL.
try:
    import immutabledict  # noqa: F401
except Exception:
    try:
        subprocess.check_call(
            [
                sys.executable, "-m", "pip", "install", "-q",
                "--force-reinstall", "--no-cache-dir",
                "--no-binary=:all:", "immutabledict"
            ],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        import immutabledict  # noqa: F401
    except Exception:
        _fallback = types.ModuleType("immutabledict")

        class _ImmutableDict(dict):
            def __setitem__(self, *args, **kwargs):
                raise TypeError("immutabledict is immutable")
            def update(self, *args, **kwargs):
                raise TypeError("immutabledict is immutable")

        def _immutabledict(*args, **kwargs):
            return _ImmutableDict(*args, **kwargs)

        _fallback.immutabledict = _immutabledict
        sys.modules["immutabledict"] = _fallback

study = optuna.create_study(
    study_name=OPTUNA_STUDY_NAME,
    storage=OPTUNA_STORAGE,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=8, multivariate=True),
    pruner=optuna.pruners.HyperbandPruner(
        min_resource=2, max_resource=TUNING_EPOCHS, reduction_factor=3
    ),
    load_if_exists=True,
)
print(f"Study ready. Previously recorded trials: {len(study.trials)}")

SQLAlchemy reinstalled without compiled extensions.
Please restart the Jupyter kernel.
Study ready. Previously recorded trials: 0


In [57]:
# Run only the remaining trials required to reach the configured total search budget.
remaining_trials = max(0, OPTUNA_TRIALS - len(study.trials))
if remaining_trials > 0:
    study.optimize(
        objective,
        n_trials=remaining_trials,
        gc_after_trial=True,
        show_progress_bar=True,
    )
else:
    print("The configured trial budget is already complete; the saved study is being reused.")

trial_states = pd.Series([trial.state.name for trial in study.trials]).value_counts()
print(f"Total trials stored: {len(study.trials)}")
display(trial_states.rename_axis("Trial state").to_frame("Count"))

  0%|          | 0/12 [00:00<?, ?it/s]

Total trials stored: 12


,Count
Trial state,
PRUNED,9
COMPLETE,3


In [58]:
# Review the best configuration and its tuning-subset validation macro F1.
best_params = study.best_params.copy()
best_trial_score = float(study.best_value)
best_trial = study.best_trial
best_trial_summary = pd.DataFrame([{
    "Best trial": best_trial.number,
    "Validation macro F1": best_trial_score,
    "Validation accuracy": best_trial.user_attrs.get("best_validation_accuracy"),
    "Validation loss": best_trial.user_attrs.get("best_validation_loss"),
    "Best epoch": best_trial.user_attrs.get("best_epoch"),
    "Trainable parameters": best_trial.user_attrs.get("parameter_count"),
}])
display(best_trial_summary.round(4))
display(pd.DataFrame([best_params]))

,Best trial,Validation macro F1,Validation accuracy,Validation loss,Best epoch,Trainable parameters
0,7,0.8165,0.8211,0.696,4,81626


,num_blocks,base_channels,kernel_size,dropout,learning_rate,weight_decay,optimizer,batch_size,label_smoothing
0,3,16,3,0.235,0.004,4.1306e-06,SGD,64,0.0451


In [59]:
# Convert all Optuna trials into a detailed table for ranking, comparison, and export.
trials_df = study.trials_dataframe(
    attrs=("number", "value", "params", "state", "duration", "user_attrs")
)
completed_trials = [trial for trial in study.trials if trial.state == optuna.trial.TrialState.COMPLETE]
pruned_trials = [trial for trial in study.trials if trial.state == optuna.trial.TrialState.PRUNED]
trials_df.to_csv(ARTIFACTS_DIR / "optuna_trials_full.csv", index=False)
ranked_trials = trials_df.loc[trials_df["state"] == "COMPLETE"].sort_values("value", ascending=False)
display(ranked_trials.head(15).round(5))
print(f"Completed: {len(completed_trials)} | Pruned: {len(pruned_trials)}")

,number,value,params_base_channels,params_batch_size,params_dropout,params_kernel_size,params_label_smoothing,params_learning_rate,params_num_blocks,params_optimizer,params_weight_decay,state,duration,user_attrs_best_epoch,user_attrs_best_validation_accuracy,user_attrs_best_validation_loss,user_attrs_parameter_count
7,7,0.8165,16,64,0.2351,3,0.0451,0.0040,3,SGD,0.0000e+00,COMPLETE,0 days 00:01:02.754919,4,0.8211,0.6960,81626
1,1,0.7401,24,256,0.4141,5,0.1213,0.0002,2,RMSprop,4.0000e-05,COMPLETE,0 days 00:01:26.613334,4,0.7507,1.1396,63170
0,0,0.7041,16,256,0.4465,3,0.0787,0.0010,2,Adam,3.5000e-04,COMPLETE,0 days 00:00:56.411482,4,0.7191,1.0762,21978


Completed: 3 | Pruned: 9


In [60]:
# Interactive optimization history and intermediate-value charts.
fig = optuna.visualization.plot_optimization_history(study)
save_plotly_figure(fig, "13_optuna_optimization_history", title="Optuna | Validation Macro-F1 Optimization History", height=680)

if len(study.trials) > 0:
    fig = optuna.visualization.plot_intermediate_values(study)
    save_plotly_figure(fig, "14_optuna_intermediate_values", title="Optuna | Trial Learning Curves and Pruning", height=680)

In [61]:
# Estimate hyperparameter importance and compare the distribution of completed trial scores.
if len(completed_trials) >= 5:
    try:
        importance_values = optuna.importance.get_param_importances(study)
        importance_df = pd.DataFrame(
            [{"Hyperparameter": key, "Importance": value}
             for key, value in importance_values.items()]
        ).sort_values("Importance", ascending=False)
        importance_df.to_csv(ARTIFACTS_DIR / "optuna_parameter_importance.csv", index=False)
        display(importance_df.round(5))
        fig = optuna.visualization.plot_param_importances(study)
        save_plotly_figure(fig, "15_optuna_hyperparameter_importance", title="Optuna | Hyperparameter Importance", height=680)
    except Exception:
        print("Parameter-importance estimation needs a broader or more varied set of completed trials.")

    try:
        fig = optuna.visualization.plot_edf(study)
        save_plotly_figure(fig, "16_optuna_empirical_distribution", title="Optuna | Empirical Distribution of Trial Scores", height=680)
    except Exception:
        print("The empirical distribution plot is unavailable for this study.")
else:
    print("Complete more trials before interpreting global hyperparameter importance.")

Complete more trials before interpreting global hyperparameter importance.


In [62]:
# Use parallel-coordinate, slice, contour, and custom trial-ranking views to understand the search.
if len(completed_trials) >= 5:
    try:
        fig = optuna.visualization.plot_parallel_coordinate(
            study, params=["learning_rate", "weight_decay", "dropout", "base_channels", "num_blocks"]
        )
        save_plotly_figure(fig, "17_optuna_parallel_coordinates", title="Optuna | Parallel-Coordinate Search Space", height=760)
    except Exception:
        print("Parallel-coordinate analysis is unavailable for the current set of trials.")

    try:
        fig = optuna.visualization.plot_slice(
            study, params=["learning_rate", "weight_decay", "dropout"]
        )
        save_plotly_figure(fig, "18_optuna_parameter_slices", title="Optuna | Hyperparameter Slice Analysis", height=760)
    except Exception:
        print("Parameter-slice analysis is unavailable for the current set of trials.")

    try:
        fig = optuna.visualization.plot_contour(study, params=["learning_rate", "weight_decay"])
        save_plotly_figure(fig, "19_optuna_learning_rate_weight_decay_contour", title="Optuna | Learning Rate × Weight Decay", height=760)
    except Exception:
        print("Contour analysis is unavailable for the current set of trials.")

trial_plot_df = pd.DataFrame([
    {"Trial": trial.number, "Validation macro F1": trial.value,
     "State": trial.state.name, "Best epoch": trial.user_attrs.get("best_epoch")}
    for trial in study.trials
])
if not trial_plot_df.empty:
    fig = px.scatter(
        trial_plot_df, x="Trial", y="Validation macro F1", color="State",
        size="Best epoch", hover_data=["Best epoch"],
        color_discrete_map={"COMPLETE": "#00A6A6", "PRUNED": "#F28E2B", "FAIL": "#E15759"},
    )
    fig.update_yaxes(title="Validation macro F1", rangemode="tozero")
    save_plotly_figure(fig, "20_optuna_trial_score_map", title="Optuna | Trial-by-Trial Score Map", height=600)

In [63]:
# Separate architecture parameters from optimizer parameters for final model construction.
final_model_config = {
    "num_blocks": int(best_params["num_blocks"]),
    "base_channels": int(best_params["base_channels"]),
    "kernel_size": int(best_params["kernel_size"]),
    "dropout": float(best_params["dropout"]),
}
final_learning_rate = float(best_params["learning_rate"])
final_weight_decay = float(best_params["weight_decay"])
final_optimizer_name = str(best_params["optimizer"])
final_batch_size = int(best_params["batch_size"])
final_label_smoothing = float(best_params["label_smoothing"])
final_criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=final_label_smoothing)

final_config_table = pd.DataFrame([
    {"Setting": "Architecture", "Selected value": str(final_model_config)},
    {"Setting": "Optimizer", "Selected value": final_optimizer_name},
    {"Setting": "Learning rate", "Selected value": final_learning_rate},
    {"Setting": "Weight decay", "Selected value": final_weight_decay},
    {"Setting": "Batch size", "Selected value": final_batch_size},
    {"Setting": "Label smoothing", "Selected value": final_label_smoothing},
])
display(final_config_table)

,Setting,Selected value
0,Architecture,"{'num_blocks': 3, 'base_channels': 16, 'kernel..."
1,Optimizer,SGD
2,Learning rate,0.004
3,Weight decay,0.0
4,Batch size,64
5,Label smoothing,0.0451


## 6. Retrain the best Optuna configuration

The top-ranked Optuna trial defines the final architecture and training recipe. The selected model is now trained on the **complete training partition** and monitored on the validation partition. Early stopping and learning-rate scheduling still use validation data only; the official test CSV remains untouched until the next section.

In [64]:
# Rebuild loaders at the batch size selected by Optuna.
final_train_loader = DataLoader(train_dataset, batch_size=final_batch_size, shuffle=True,
                                num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
final_val_loader = DataLoader(val_dataset, batch_size=final_batch_size, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
print(f"Final training batch size: {final_batch_size}")

Final training batch size: 64


In [65]:
# Train the tuned CNN on the complete training split with the best Optuna-selected loss and optimizer.
final_model = FashionCNN(**final_model_config).to(DEVICE)
final_optimizer = create_optimizer(final_model, final_learning_rate,
                                   final_weight_decay, final_optimizer_name)
final_history, best_final_val_accuracy = fit_model(
    final_model, final_train_loader, final_val_loader, final_optimizer,
    epochs=FINAL_EPOCHS, patience=EARLY_STOPPING_PATIENCE,
    loss_function=final_criterion,
)
print(f"Best full-validation accuracy: {best_final_val_accuracy:.4f}")

Epoch 01/24 | train acc 0.6502 | val acc 0.7979 | val loss 0.7401
Epoch 02/24 | train acc 0.7933 | val acc 0.8298 | val loss 0.6768
Epoch 03/24 | train acc 0.8324 | val acc 0.8736 | val loss 0.5867
Epoch 04/24 | train acc 0.8526 | val acc 0.8820 | val loss 0.5634
Epoch 05/24 | train acc 0.8652 | val acc 0.8864 | val loss 0.5498
Epoch 06/24 | train acc 0.8725 | val acc 0.8947 | val loss 0.5292
Epoch 07/24 | train acc 0.8778 | val acc 0.8938 | val loss 0.5301
Epoch 08/24 | train acc 0.8819 | val acc 0.8942 | val loss 0.5249
Epoch 09/24 | train acc 0.8884 | val acc 0.9018 | val loss 0.5054
Epoch 10/24 | train acc 0.8897 | val acc 0.9081 | val loss 0.4921
Epoch 11/24 | train acc 0.8933 | val acc 0.9086 | val loss 0.4927
Epoch 12/24 | train acc 0.8972 | val acc 0.9066 | val loss 0.4950
Epoch 13/24 | train acc 0.8982 | val acc 0.9098 | val loss 0.4881
Epoch 14/24 | train acc 0.9015 | val acc 0.9137 | val loss 0.4812
Epoch 15/24 | train acc 0.9032 | val acc 0.9152 | val loss 0.4784
Epoch 16/2

In [66]:
# Display final training history.
display(final_history.round(4))

,epoch,train_loss,train_accuracy,val_loss,val_accuracy,learning_rate
0,1,1.1214,0.6502,0.7401,0.7979,0.004
1,2,0.7782,0.7933,0.6768,0.8298,0.004
2,3,0.6952,0.8324,0.5867,0.8736,0.004
3,4,0.6508,0.8526,0.5634,0.8820,0.004
4,5,0.6202,0.8652,0.5498,0.8864,0.004
5,6,0.6007,0.8725,0.5292,0.8947,0.004
6,7,0.5869,0.8778,0.5301,0.8938,0.004
7,8,0.5726,0.8819,0.5249,0.8942,0.004
8,9,0.5629,0.8884,0.5054,0.9018,0.004
9,10,0.5559,0.8897,0.4921,0.9081,0.004


In [67]:
# Plot final loss, accuracy, and learning-rate trajectories using Plotly subplots.
fig = make_subplots(rows=1, cols=3, subplot_titles=["Loss", "Accuracy", "Learning rate"])
for name, column, color in [("Train", "train_loss", "#5B5FDE"), ("Validation", "val_loss", "#00A6A6")]:
    fig.add_trace(go.Scatter(x=final_history["epoch"], y=final_history[column], mode="lines+markers",
                             name=f"{name} loss", line=dict(color=color, width=3)), row=1, col=1)
for name, column, color in [("Train", "train_accuracy", "#5B5FDE"), ("Validation", "val_accuracy", "#00A6A6")]:
    fig.add_trace(go.Scatter(x=final_history["epoch"], y=final_history[column], mode="lines+markers",
                             name=f"{name} accuracy", line=dict(color=color, width=3)), row=1, col=2)
fig.add_trace(go.Scatter(x=final_history["epoch"], y=final_history["learning_rate"],
                         mode="lines+markers", name="Learning rate", line=dict(color="#F28E2B", width=3)), row=1, col=3)
for column in [1, 2, 3]:
    fig.update_xaxes(title_text="Epoch", row=1, col=column)
fig.update_yaxes(title_text="Loss", row=1, col=1)
fig.update_yaxes(title_text="Accuracy", range=[0, 1], row=1, col=2)
fig.update_yaxes(title_text="Learning rate", type="log", row=1, col=3)
save_plotly_figure(fig, "21_final_training_diagnostics", title="Optuna-Tuned CNN | Final Training Diagnostics", height=560, width=1600)

In [68]:
# Define a compact wrapper for predictions on any DataLoader.
def get_predictions(model, loader):
    return predict_loader(model, loader)

print("Prediction helper is ready.")

Prediction helper is ready.


## 7. Official test evaluation

The separate test CSV is used here for the first time. The following metrics, confusion matrices, and class-level reports provide the final generalization estimate.

In [69]:
# Final evaluation on the untouched official test CSV.
y_test_true, y_test_pred, test_probabilities = get_predictions(final_model, test_loader)
test_metrics = classification_metrics(y_test_true, y_test_pred)
print("Official test evaluation completed.")
display(pd.DataFrame([test_metrics]).T.rename(columns={0: "Official test"}))

Official test evaluation completed.


,Official test
accuracy,0.9281
macro_precision,0.9288
macro_recall,0.9281
macro_f1,0.9282
weighted_f1,0.9282


In [70]:
# Generate a detailed precision, recall, F1, and support report.
report_dict = classification_report(
    y_test_true, y_test_pred,
    labels=CLASS_IDS,
    target_names=[CLASS_NAMES[i] for i in CLASS_IDS],
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).T
display(report_df.round(4))

,precision,recall,f1-score,support
T-shirt/top,0.8806,0.8780,0.8793,1000.0000
Trouser,0.9940,0.9880,0.9910,1000.0000
Pullover,0.9335,0.8560,0.8931,1000.0000
Dress,0.9188,0.9500,0.9341,1000.0000
Coat,0.8900,0.9060,0.8979,1000.0000
Sandal,0.9929,0.9730,0.9828,1000.0000
Shirt,0.7692,0.7930,0.7809,1000.0000
Sneaker,0.9630,0.9640,0.9635,1000.0000
Bag,0.9822,0.9910,0.9866,1000.0000
Ankle boot,0.9637,0.9820,0.9728,1000.0000


In [71]:
# Compare the tuned model against the baseline validation reference.
model_comparison = pd.DataFrame([
    {"Model": "Baseline CNN (validation)", **baseline_metrics},
    {"Model": "Tuned CNN (official test)", **test_metrics},
])
display(model_comparison.set_index("Model").round(4))

,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1
Model,,,,,
Baseline CNN (validation),0.8494,0.8598,0.8494,0.8522,0.8522
Tuned CNN (official test),0.9281,0.9288,0.9281,0.9282,0.9282


In [72]:
# Calculate the official test confusion matrix.
test_cm = confusion_matrix(y_test_true, y_test_pred, labels=CLASS_IDS)
assert test_cm.shape == (NUM_CLASSES, NUM_CLASSES)
print(f"Correct predictions: {int(np.trace(test_cm)):,} / {len(y_test_true):,}")

Correct predictions: 9,281 / 10,000


In [73]:
# Plot the official test confusion matrix with interactive counts and hover details.
fig = go.Figure(go.Heatmap(
    z=test_cm, x=class_axis_labels, y=class_axis_labels, colorscale="Turbo",
    text=test_cm, texttemplate="%{text}", colorbar=dict(title="Samples"),
    hovertemplate="True: %{y}<br>Predicted: %{x}<br>Count: %{z}<extra></extra>"
))
fig.update_layout(xaxis_title="Predicted class", yaxis_title="True class")
fig.update_xaxes(tickangle=-30)
fig.update_yaxes(autorange="reversed")
save_plotly_figure(fig, "22_official_test_confusion_matrix", title="Optuna-Tuned CNN | Official Test Confusion Matrix", height=780, width=1150)

In [74]:
# Row-normalized confusion matrix makes class-wise error patterns easier to compare.
row_sums = np.maximum(test_cm.sum(axis=1, keepdims=True), 1)
normalized_cm = test_cm / row_sums
fig = go.Figure(go.Heatmap(
    z=normalized_cm, x=class_axis_labels, y=class_axis_labels,
    colorscale="Viridis", zmin=0, zmax=1,
    text=np.round(normalized_cm, 2), texttemplate="%{text:.2f}",
    colorbar=dict(title="Row share"),
    hovertemplate="True: %{y}<br>Predicted: %{x}<br>Row share: %{z:.3f}<extra></extra>"
))
fig.update_layout(xaxis_title="Predicted class", yaxis_title="True class")
fig.update_xaxes(tickangle=-30)
fig.update_yaxes(autorange="reversed")
save_plotly_figure(fig, "23_normalized_confusion_matrix", title="Row-Normalized Confusion Matrix", height=780, width=1150)

In [75]:
# Extract per-class precision, recall, F1, and support.
per_class_metrics = report_df.loc[[CLASS_NAMES[i] for i in CLASS_IDS],
                                  ["precision", "recall", "f1-score", "support"]].copy()
per_class_metrics.index.name = "Class"
display(per_class_metrics.round(4))

,precision,recall,f1-score,support
Class,,,,
T-shirt/top,0.8806,0.878,0.8793,1000.0
Trouser,0.9940,0.988,0.9910,1000.0
Pullover,0.9335,0.856,0.8931,1000.0
Dress,0.9188,0.950,0.9341,1000.0
Coat,0.8900,0.906,0.8979,1000.0
Sandal,0.9929,0.973,0.9828,1000.0
Shirt,0.7692,0.793,0.7809,1000.0
Sneaker,0.9630,0.964,0.9635,1000.0
Bag,0.9822,0.991,0.9866,1000.0


In [76]:
# Plot class-wise F1 and recall as side-by-side interactive indicators.
metric_plot = per_class_metrics.reset_index().melt(
    id_vars="Class", value_vars=["f1-score", "recall"],
    var_name="Metric", value_name="Score"
)
fig = px.bar(
    metric_plot, x="Class", y="Score", color="Metric", barmode="group",
    color_discrete_map={"f1-score": "#5B5FDE", "recall": "#00A6A6"},
    range_y=[0, 1.05], labels={"Score": "Metric score"},
)
fig.update_xaxes(tickangle=-25, showgrid=False)
save_plotly_figure(fig, "24_per_class_f1_recall", title="Per-Class F1 and Recall", height=580)

## 8. Diagnostic and confidence analysis

ROC and precision–recall curves provide threshold-independent views. Confidence plots and misclassification galleries help reveal where the model is uncertain or confidently wrong.

In [77]:
# Convert labels to one-vs-rest indicators for multiclass ROC analysis.
y_test_binary = label_binarize(y_test_true, classes=CLASS_IDS)
roc_records = {}
for class_id in CLASS_IDS:
    fpr, tpr, _ = roc_curve(y_test_binary[:, class_id], test_probabilities[:, class_id])
    roc_records[class_id] = {"fpr": fpr, "tpr": tpr, "auc": auc(fpr, tpr)}
macro_auc = float(np.mean([roc_records[i]["auc"] for i in CLASS_IDS]))
print(f"Macro one-vs-rest ROC AUC: {macro_auc:.4f}")

Macro one-vs-rest ROC AUC: 0.9961


In [78]:
# Plot one-vs-rest ROC curves for all fashion categories.
fig = go.Figure()
for class_id, color in zip(CLASS_IDS, PLOTLY_COLORS):
    record = roc_records[class_id]
    fig.add_trace(go.Scatter(
        x=record["fpr"], y=record["tpr"], mode="lines",
        name=f"{CLASS_NAMES[class_id]} (AUC={record['auc']:.3f})",
        line=dict(color=color, width=2.5),
    ))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Chance",
                         line=dict(color="#64748B", dash="dash", width=2)))
fig.update_layout(xaxis_title="False positive rate", yaxis_title="True positive rate",
                  legend=dict(orientation="h", y=-0.23))
fig.update_xaxes(range=[0, 1])
fig.update_yaxes(range=[0, 1])
save_plotly_figure(fig, "25_multiclass_roc_curves", title="Multiclass ROC Curves | Official Test", height=720, width=1300)

In [79]:
# Compute and plot precision-recall curves for the ten one-vs-rest class tasks.
fig = go.Figure()
for class_id, color in zip(CLASS_IDS, PLOTLY_COLORS):
    precision_values, recall_values, _ = precision_recall_curve(
        y_test_binary[:, class_id], test_probabilities[:, class_id]
    )
    fig.add_trace(go.Scatter(
        x=recall_values, y=precision_values, mode="lines",
        name=CLASS_NAMES[class_id], line=dict(color=color, width=2.5),
    ))
fig.update_layout(xaxis_title="Recall", yaxis_title="Precision",
                  legend=dict(orientation="h", y=-0.23))
fig.update_xaxes(range=[0, 1])
fig.update_yaxes(range=[0, 1])
save_plotly_figure(fig, "26_multiclass_precision_recall_curves", title="Multiclass Precision–Recall Curves", height=720, width=1300)

In [80]:
# Inspect confidence distributions for correct and incorrect predictions.
test_confidence = test_probabilities.max(axis=1)
test_correct = y_test_pred == y_test_true
confidence_frame = pd.DataFrame({
    "Confidence": test_confidence,
    "Prediction status": np.where(test_correct, "Correct", "Incorrect"),
})
fig = px.histogram(
    confidence_frame, x="Confidence", color="Prediction status", nbins=30,
    histnorm="probability density", barmode="overlay", opacity=0.72,
    color_discrete_map={"Correct": "#00A676", "Incorrect": "#E15759"},
)
fig.update_xaxes(range=[0, 1])
save_plotly_figure(fig, "27_prediction_confidence_distribution", title="Prediction Confidence | Correct vs Incorrect", height=570)

In [81]:
# Build a top-label reliability table for a multiclass calibration diagnostic.
confidence_bins = np.linspace(0.0, 1.0, 11)
reliability_rows = []
for lower, upper in zip(confidence_bins[:-1], confidence_bins[1:]):
    selected = (test_confidence >= lower) & (test_confidence < upper if upper < 1 else test_confidence <= upper)
    if selected.any():
        reliability_rows.append({
            "Mean confidence": float(test_confidence[selected].mean()),
            "Observed accuracy": float(test_correct[selected].mean()),
            "Count": int(selected.sum()),
        })
reliability_df = pd.DataFrame(reliability_rows)
display(reliability_df.round(4))

,Mean confidence,Observed accuracy,Count
0,0.2685,0.2857,7
1,0.3522,0.4198,81
2,0.4587,0.4684,237
3,0.5513,0.5858,367
4,0.6500,0.6930,430
5,0.7524,0.7964,550
6,0.8571,0.9000,880
7,0.9712,0.9923,7448


In [82]:
# Plot the reliability curve and the ideal calibration diagonal.
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=reliability_df["Mean confidence"], y=reliability_df["Observed accuracy"],
    mode="lines+markers", name="Model", line=dict(color="#7C4DFF", width=3),
    marker=dict(size=10, color="#00A6A6"),
    customdata=reliability_df[["Count"]],
    hovertemplate="Mean confidence: %{x:.3f}<br>Observed accuracy: %{y:.3f}<br>Count: %{customdata[0]}<extra></extra>"
))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Perfect calibration",
                         line=dict(color="#64748B", dash="dash", width=2)))
fig.update_layout(xaxis_title="Mean predicted confidence", yaxis_title="Observed accuracy")
fig.update_xaxes(range=[0, 1])
fig.update_yaxes(range=[0, 1], scaleanchor="x", scaleratio=1)
save_plotly_figure(fig, "28_calibration_reliability_curve", title="Top-Label Calibration", height=680, width=850)

In [83]:
# Helper for a single normalized image prediction.
def predict_single_image(image_uint8, model=final_model):
    image_tensor = torch.as_tensor(image_uint8, dtype=torch.float32).reshape(1, 1, 28, 28) / 255.0
    image_tensor = (image_tensor - PIXEL_MEAN) / PIXEL_STD
    model.eval()
    with torch.no_grad():
        probabilities = torch.softmax(model(image_tensor.to(DEVICE)), dim=1)[0].cpu().numpy()
    predicted_id = int(probabilities.argmax())
    return predicted_id, probabilities

In [84]:
# Display a grid of correctly classified official test samples.
correct_indices = np.flatnonzero(test_correct)
chosen_correct = rng.choice(correct_indices, size=min(20, len(correct_indices)), replace=False)
fig = make_subplots(rows=4, cols=5, subplot_titles=[
    f"True: {CLASS_NAMES[y_test_true[idx]]}<br>Pred: {CLASS_NAMES[y_test_pred[idx]]}" for idx in chosen_correct
], vertical_spacing=0.07, horizontal_spacing=0.035)
for position, idx in enumerate(chosen_correct):
    row, col = divmod(position, 5)
    image = X_test_raw[idx].reshape(28, 28)
    fig.add_trace(go.Heatmap(z=image, colorscale="Gray", zmin=0, zmax=255,
                             showscale=False, hoverinfo="skip"), row=row + 1, col=col + 1)
    fig.update_xaxes(visible=False, row=row + 1, col=col + 1)
    fig.update_yaxes(visible=False, autorange="reversed", row=row + 1, col=col + 1)
save_plotly_figure(fig, "29_correct_test_predictions", title="Correct Official Test Predictions", height=850, width=1250)

In [85]:
# Display a grid of misclassified official test samples.
incorrect_indices = np.flatnonzero(~test_correct)
chosen_incorrect = rng.choice(incorrect_indices, size=min(20, len(incorrect_indices)), replace=False)
fig = make_subplots(rows=4, cols=5, subplot_titles=[
    f"True: {CLASS_NAMES[y_test_true[idx]]}<br>Pred: {CLASS_NAMES[y_test_pred[idx]]}" for idx in chosen_incorrect
], vertical_spacing=0.07, horizontal_spacing=0.035)
for position, idx in enumerate(chosen_incorrect):
    row, col = divmod(position, 5)
    image = X_test_raw[idx].reshape(28, 28)
    fig.add_trace(go.Heatmap(z=image, colorscale="Gray", zmin=0, zmax=255,
                             showscale=False, hoverinfo="skip"), row=row + 1, col=col + 1)
    fig.update_xaxes(visible=False, row=row + 1, col=col + 1)
    fig.update_yaxes(visible=False, autorange="reversed", row=row + 1, col=col + 1)
save_plotly_figure(fig, "30_misclassified_test_predictions", title="Misclassified Official Test Samples", height=850, width=1250)

In [86]:
# Inspect the most confident mistakes; these are high-confidence failure cases.
if len(incorrect_indices):
    hardest_errors = incorrect_indices[np.argsort(test_confidence[incorrect_indices])[::-1][:12]]
    fig = make_subplots(rows=3, cols=4, subplot_titles=[
        f"{CLASS_NAMES[y_test_true[idx]]} → {CLASS_NAMES[y_test_pred[idx]]}<br>Confidence: {test_confidence[idx]:.2f}"
        for idx in hardest_errors
    ], vertical_spacing=0.08, horizontal_spacing=0.04)
    for position, idx in enumerate(hardest_errors):
        row, col = divmod(position, 4)
        fig.add_trace(go.Heatmap(z=X_test_raw[idx].reshape(28, 28), colorscale="Gray",
                                 zmin=0, zmax=255, showscale=False, hoverinfo="skip"), row=row + 1, col=col + 1)
        fig.update_xaxes(visible=False, row=row + 1, col=col + 1)
        fig.update_yaxes(visible=False, autorange="reversed", row=row + 1, col=col + 1)
    save_plotly_figure(fig, "31_high_confidence_errors", title="Highest-Confidence Misclassifications", height=780, width=1250)
else:
    print("No misclassifications were found in the official test set.")

In [87]:
# Compare per-class sample counts and correct prediction rates.
class_accuracy_rows = []
for class_id in CLASS_IDS:
    class_mask = y_test_true == class_id
    class_accuracy_rows.append({
        "Class": CLASS_NAMES[class_id],
        "Samples": int(class_mask.sum()),
        "Accuracy": float(test_correct[class_mask].mean()) if class_mask.any() else np.nan,
    })
class_accuracy_df = pd.DataFrame(class_accuracy_rows)
display(class_accuracy_df.round(4))

,Class,Samples,Accuracy
0,T-shirt/top,1000,0.878
1,Trouser,1000,0.988
2,Pullover,1000,0.856
3,Dress,1000,0.950
4,Coat,1000,0.906
5,Sandal,1000,0.973
6,Shirt,1000,0.793
7,Sneaker,1000,0.964
8,Bag,1000,0.991
9,Ankle boot,1000,0.982


In [88]:
# Plot per-class accuracy with sample counts annotated.
fig = px.bar(
    class_accuracy_df, x="Class", y="Accuracy", color="Accuracy",
    color_continuous_scale="Viridis", range_color=[0, 1],
    hover_data=["Samples"], text=class_accuracy_df["Accuracy"].map(lambda value: f"{value:.1%}"),
)
fig.update_traces(textposition="outside")
fig.update_xaxes(tickangle=-25, showgrid=False)
fig.update_yaxes(range=[0, 1.08], tickformat=".0%")
save_plotly_figure(fig, "32_official_test_accuracy_by_class", title="Official Test Accuracy by Class", height=580)

## 9. Model interpretability

Inspect learned first-layer filters and intermediate feature activations. Grad-CAM provides a qualitative indication of image regions that contribute to a selected class score.

In [89]:
# Visualize learned first-layer convolutional filters with interactive heatmap subplots.
first_conv = next(layer for layer in final_model.features if isinstance(layer, nn.Conv2d))
filters = first_conv.weight.detach().cpu().numpy()
filter_count = min(24, filters.shape[0])
fig = make_subplots(rows=4, cols=6, subplot_titles=[f"Filter {i + 1}" for i in range(filter_count)],
                    vertical_spacing=0.08, horizontal_spacing=0.04)
for filter_index in range(filter_count):
    row, col = divmod(filter_index, 6)
    fig.add_trace(go.Heatmap(z=filters[filter_index, 0], colorscale="RdBu",
                             showscale=False, hoverinfo="skip"), row=row + 1, col=col + 1)
    fig.update_xaxes(visible=False, row=row + 1, col=col + 1)
    fig.update_yaxes(visible=False, autorange="reversed", row=row + 1, col=col + 1)
save_plotly_figure(fig, "33_learned_convolution_filters", title="Learned First-Layer Convolutional Filters", height=760, width=1200)

In [90]:
# Capture activation maps from the final convolutional layer for one test image.
activation_cache = {}
def capture_activation(module, module_input, module_output):
    activation_cache["feature_maps"] = module_output.detach().cpu()

activation_handle = final_model.target_layer.register_forward_hook(capture_activation)
activation_sample = next(iter(test_loader))[0][:1].to(DEVICE)
final_model.eval()
with torch.no_grad():
    _ = final_model(activation_sample)
activation_handle.remove()
feature_maps = activation_cache["feature_maps"][0]
print(f"Captured feature maps: {tuple(feature_maps.shape)}")

Captured feature maps: (64, 7, 7)


In [91]:
# Plot a selection of learned feature maps in an interactive grid.
feature_count = min(24, feature_maps.shape[0])
fig = make_subplots(rows=4, cols=6, subplot_titles=[f"Map {i + 1}" for i in range(feature_count)],
                    vertical_spacing=0.08, horizontal_spacing=0.04)
for feature_index in range(feature_count):
    row, col = divmod(feature_index, 6)
    fig.add_trace(go.Heatmap(z=feature_maps[feature_index].numpy(), colorscale="Viridis",
                             showscale=False, hoverinfo="skip"), row=row + 1, col=col + 1)
    fig.update_xaxes(visible=False, row=row + 1, col=col + 1)
    fig.update_yaxes(visible=False, autorange="reversed", row=row + 1, col=col + 1)
save_plotly_figure(fig, "34_intermediate_feature_maps", title="Intermediate CNN Feature Activations", height=760, width=1200)

In [92]:
# Grad-CAM helper: highlight image regions that influence a selected class score.
def generate_gradcam(model, image_tensor, class_index=None):
    captured = {"activations": None, "gradients": None}
    def forward_hook(module, module_input, module_output):
        captured["activations"] = module_output
        module_output.retain_grad()
    forward_handle = model.target_layer.register_forward_hook(forward_hook)
    model.eval()
    model.zero_grad(set_to_none=True)
    logits = model(image_tensor.to(DEVICE))
    if class_index is None:
        class_index = int(logits.argmax(dim=1).item())
    logits[:, class_index].sum().backward()
    activations = captured["activations"]
    gradients = activations.grad
    weights = gradients.mean(dim=(2, 3), keepdim=True)
    heatmap = (weights * activations).sum(dim=1, keepdim=True).relu()
    heatmap = F.interpolate(heatmap, size=(28, 28), mode="bilinear", align_corners=False)
    heatmap = heatmap[0, 0].detach().cpu().numpy()
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    forward_handle.remove()
    return heatmap, class_index

In [93]:
# Generate a Grad-CAM explanation for a representative official test example.
gradcam_index = int(correct_indices[0]) if len(correct_indices) else 0
gradcam_image = torch.as_tensor(X_test_raw[gradcam_index], dtype=torch.float32).view(1, 1, 28, 28) / 255.0
gradcam_image = (gradcam_image - PIXEL_MEAN) / PIXEL_STD
gradcam_map, gradcam_class = generate_gradcam(final_model, gradcam_image)
print(f"Grad-CAM target category: {CLASS_NAMES[gradcam_class]}")

Grad-CAM target category: T-shirt/top


In [94]:
# Compare the original test image, its Grad-CAM heatmap, and the overlay.
original_image = X_test_raw[gradcam_index].reshape(28, 28)
fig = make_subplots(rows=1, cols=3, subplot_titles=["Original image", "Grad-CAM heatmap", "Activation overlay"])
fig.add_trace(go.Heatmap(z=original_image, colorscale="Gray", zmin=0, zmax=255, showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=gradcam_map, colorscale="Inferno", zmin=0, zmax=1,
                         colorbar=dict(title="Activation", x=0.65)), row=1, col=2)
fig.add_trace(go.Heatmap(z=original_image, colorscale="Gray", zmin=0, zmax=255, showscale=False), row=1, col=3)
fig.add_trace(go.Heatmap(z=gradcam_map, colorscale="Turbo", zmin=0, zmax=1,
                         opacity=0.52, showscale=False), row=1, col=3)
for column in [1, 2, 3]:
    fig.update_xaxes(visible=False, row=1, col=column)
    fig.update_yaxes(visible=False, autorange="reversed", scaleanchor=f"x{column if column > 1 else ''}", row=1, col=column)
save_plotly_figure(fig, "35_gradcam_explanation", title="Grad-CAM | Visual Model Explanation", height=520, width=1400)

## 10. Export, reproducibility, and experiment audit

The final section saves the tuned model, selected hyperparameters, Optuna trial history, persistent study database, metrics, and predictions. Plotly charts are exported separately into `images/`. No generated file locations are printed into the notebook output.

In [95]:
# Build a tidy prediction table with true class, predicted class, and confidence.
prediction_table = pd.DataFrame({
    "sample_index": np.arange(len(y_test_true)),
    "true_label": y_test_true,
    "true_class": [CLASS_NAMES[int(i)] for i in y_test_true],
    "predicted_label": y_test_pred,
    "predicted_class": [CLASS_NAMES[int(i)] for i in y_test_pred],
    "confidence": test_confidence,
    "correct": test_correct,
})
display(prediction_table.head(10))

,sample_index,true_label,true_class,predicted_label,predicted_class,confidence,correct
0,0,0,T-shirt/top,0,T-shirt/top,0.9664,True
1,1,1,Trouser,1,Trouser,0.9807,True
2,2,2,Pullover,2,Pullover,0.9287,True
3,3,2,Pullover,2,Pullover,0.4676,True
4,4,3,Dress,3,Dress,0.7984,True
5,5,2,Pullover,6,Shirt,0.7764,False
6,6,8,Bag,8,Bag,0.9792,True
7,7,6,Shirt,6,Shirt,0.8731,True
8,8,5,Sandal,5,Sandal,0.9933,True
9,9,0,T-shirt/top,0,T-shirt/top,0.9908,True


In [96]:
# Save the official test prediction table and per-class classification report.
prediction_table.to_csv(ARTIFACTS_DIR / "official_test_predictions.csv", index=False)
report_df.to_csv(ARTIFACTS_DIR / "classification_report.csv")
per_class_metrics.to_csv(ARTIFACTS_DIR / "per_class_metrics.csv")
print("Prediction and classification-report artifacts saved.")

Prediction and classification-report artifacts saved.


In [97]:
# Save training history, Optuna trial records, confusion matrices, and key metrics.
final_history.to_csv(ARTIFACTS_DIR / "final_training_history.csv", index=False)
baseline_history.to_csv(ARTIFACTS_DIR / "baseline_training_history.csv", index=False)
trials_df.to_csv(ARTIFACTS_DIR / "optuna_trials_full.csv", index=False)
pd.DataFrame(test_cm, index=class_axis_labels, columns=class_axis_labels).to_csv(
    ARTIFACTS_DIR / "official_test_confusion_matrix.csv"
)
with open(ARTIFACTS_DIR / "evaluation_metrics.json", "w", encoding="utf-8") as file:
    json.dump({**test_metrics, "macro_ovr_roc_auc": macro_auc,
               "best_full_validation_accuracy": best_final_val_accuracy,
               "best_optuna_validation_macro_f1": best_trial_score,
               "completed_trials": len(completed_trials),
               "pruned_trials": len(pruned_trials)}, file, indent=2)
print("Training and Optuna experiment artifacts saved.")

Training and Optuna experiment artifacts saved.


In [98]:
# Save a reusable checkpoint and the Optuna-selected hyperparameter configuration.
checkpoint = {
    "state_dict": final_model.state_dict(),
    "model_config": final_model_config,
    "pixel_mean": PIXEL_MEAN,
    "pixel_std": PIXEL_STD,
    "class_names": CLASS_NAMES,
    "test_metrics": test_metrics,
    "best_validation_accuracy": best_final_val_accuracy,
    "best_optuna_validation_macro_f1": best_trial_score,
    "seed": SEED,
}
torch.save(checkpoint, ARTIFACTS_DIR / "fashion_mnist_cnn_best.pth")
with open(ARTIFACTS_DIR / "best_hyperparameters.json", "w", encoding="utf-8") as file:
    json.dump({**best_params, "best_tuning_validation_macro_f1": best_trial_score}, file, indent=2)
print("Model checkpoint and selected hyperparameters saved.")

Model checkpoint and selected hyperparameters saved.


In [99]:
# Export a TorchScript inference model and verify it with a sample batch.
final_model.eval()
example_input = torch.zeros(1, 1, 28, 28, dtype=torch.float32, device=DEVICE)
traced_model = torch.jit.trace(final_model, example_input)
traced_model.save(str(ARTIFACTS_DIR / "fashion_mnist_cnn_torchscript.pt"))
with torch.no_grad():
    exported_logits = traced_model(example_input)
assert tuple(exported_logits.shape) == (1, NUM_CLASSES)
print("TorchScript export smoke test passed.")

TorchScript export smoke test passed.


In [100]:
# Final experiment summary: show model quality and the effectiveness of the Optuna search.
final_summary = pd.DataFrame({
    "Metric": ["Official test accuracy", "Macro precision", "Macro recall",
               "Macro F1", "Weighted F1", "Macro one-vs-rest ROC AUC",
               "Best full-validation accuracy", "Best Optuna validation macro F1",
               "Completed Optuna trials", "Pruned Optuna trials"],
    "Value": [test_metrics["accuracy"], test_metrics["macro_precision"],
              test_metrics["macro_recall"], test_metrics["macro_f1"],
              test_metrics["weighted_f1"], macro_auc, best_final_val_accuracy,
              best_trial_score, len(completed_trials), len(pruned_trials)],
})
display(final_summary.round(4))
print("Fashion-MNIST Optuna hyperparameter optimization workflow completed.")

,Metric,Value
0,Official test accuracy,0.9281
1,Macro precision,0.9288
2,Macro recall,0.9281
3,Macro F1,0.9282
4,Weighted F1,0.9282
5,Macro one-vs-rest ROC AUC,0.9961
6,Best full-validation accuracy,0.9210
7,Best Optuna validation macro F1,0.8165
8,Completed Optuna trials,3.0000
9,Pruned Optuna trials,9.0000


Fashion-MNIST Optuna hyperparameter optimization workflow completed.
